In [ ]:
# Purpose - looks at the horizontal cut of the wave over time

findex = 280 #260 #-7 #140 #230 #190 #230 #140 #-7 

step_choice = "all" #"penultimate" #"last" #"last" #all

In [ ]:
# Imports and Folder Selection

%load_ext autoreload
%autoreload 2

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq
import warnings; warnings.filterwarnings('ignore')
import sys
sys.path.append('../v2_proc_files')
from useful_funcs import *
from TristanRun import TristanRun
from FieldLineHelper import FieldLineHelper 

plt.close('all')

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq

cc = run.cc
B0 = run.B0

tstep_choices = {"first": [run.steps[0]], "last": [run.steps[-1]], "all": run.steps[:-1], "penultimate": [run.steps[-2]],
                 "last_two": run.steps[-2:]}

my_steps = tstep_choices[step_choice]
#my_steps = list(range(85, 681, 5))
#my_steps = [85]
print(len(my_steps))
my_steps = range(27)
#my_steps = run.steps[-6:]

#boilerplate = run.boilerplate + f" tstep = {my_steps[0]}"

In [ ]:
d = run.config_data(my_steps, "fields", includes_pressure = False)

In [ ]:
def low_pass_filter(da, dim='x', cutoff_freq=0.1):
    signal = da.values
    n = signal.size
    dx = da[dim].values[1] - da[dim].values[0]  # Sampling interval
    
    freqs = fftfreq(n, d=dx)
    

    spec = fft(signal)
    spec[np.abs(freqs) > cutoff_freq] = 0
    
    filtered_signal = ifft(spec).real
    
    return da.copy(data=filtered_signal)


def split_and_align_halves(ds, dim='x', center_val=None, match_length=True):
    if center_val is None:
        center_val = ds[dim].max().item() / 2
        
    # Split the halves
    left_half = ds.sel({dim: slice(None, center_val)})
    right_half = ds.sel({dim: slice(center_val, None)})
    
    # Flip the left half and reset its coordinate to start from 0
    # (Distance increasing away from the center)
    left_half_flipped = left_half.isel({dim: slice(None, None, -1)})
    left_half_flipped.coords[dim] = ds[dim].sel({dim: slice(None, len(left_half_flipped[dim]) - 1, 1)}) - ds[dim].min()
    
    # Reset right half coordinate to start from 0
    right_half.coords[dim] = right_half[dim] - center_val
    
    if match_length:
        # Ensure both arrays have the same number of points for easier math/plotting
        min_len = min(len(left_half_flipped[dim]), len(right_half[dim]))
        left_half_flipped = left_half_flipped.isel({dim: slice(0, min_len)})
        right_half = right_half.isel({dim: slice(0, min_len)})
        
    return right_half, left_half_flipped



In [ ]:
ts = 12
var = 'ez'
ylims = [-0.03, 0.04] #[0.5, 2] #
freq_threshold = 0.1

In [ ]:
counter = 0

x = 0 / 0

for ts in my_steps:
    plt.close('all')

    rect = [0, run.xlim, run.ylim / 2 - 5, run.ylim / 2 + 5]

    roi = fq_cut(d.fields, ts, rect)
    xcut_y = run.ylim / 2

    midstrip = fq_xcut(roi, xcut_y)

    midstripr, midstripl = split_and_align_halves(midstrip, dim='x', center_val=run.xlim / 2, match_length=True)

    midstripr_fltd = low_pass_filter(midstripr[var], dim='x', cutoff_freq=freq_threshold)
    midstripl_fltd = low_pass_filter(midstripl[var], dim='x', cutoff_freq=freq_threshold)

    strip_names = ['right', 'left']
    strips = [midstripr_fltd, midstripl_fltd]

    plt.title(run.boilerplate + f"\n tstep = {ts}")

    for strip, name in zip(strips, strip_names):
        plt.plot(strip.x, strip, label=name)

    plt.ylim(ylims[0], ylims[1])
    plt.axhline(y=0, color='k', linestyle='--', alpha=0.5)
    plt.xlabel('comp from midline')
    plt.ylabel(var)
    plt.legend()
    plt.show()

    save_movie_frame(counter) 
    counter += 1


In [ ]:
import xarray as xr

# 1. Initialize lists to hold the snapshots
left_list = []
right_list = []

for ts in my_steps:
    # ... [Your existing extraction code] ...
    roi = fq_cut(d.fields, ts, rect)
    midstrip = fq_xcut(roi, run.ylim / 2)
    
    midstripr, midstripl = split_and_align_halves(
        midstrip, dim='x', center_val=run.xlim / 2, match_length=True
    )

    # Filter and ensure we have a 'time' coordinate for concatenation
    ml_fltd = low_pass_filter(midstripl[var], dim='x', cutoff_freq=freq_threshold).expand_dims(time=[ts])
    mr_fltd = low_pass_filter(midstripr[var], dim='x', cutoff_freq=freq_threshold).expand_dims(time=[ts])

    left_list.append(ml_fltd)
    right_list.append(mr_fltd)

# 2. Concatenate into 2D DataArrays (Time, X)
all_left = xr.concat(left_list, dim='time')
all_right = xr.concat(right_list, dim='time')

# 3. Now we can apply the Hilbert Transform across the whole block
# We use .values to compute the magnitude of the envelope
def get_envelope_2d(da):
    from scipy.signal import hilbert
    # Apply hilbert along the spatial dimension (axis=-1 usually)
    return xr.DataArray(
        np.abs(hilbert(da.values, axis=-1)), 
        coords=da.coords, 
        dims=da.dims
    )

env_left = get_envelope_2d(all_left)
env_right = get_envelope_2d(all_right)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert
import xarray as xr

# 1. Compute the 2D envelopes
# We apply hilbert along the 'x' dimension (axis=1)
env_left = xr.DataArray(np.abs(hilbert(all_left.values, axis=1)), 
                        coords=all_left.coords, dims=all_left.dims)
env_right = xr.DataArray(np.abs(hilbert(all_right.values, axis=1)), 
                         coords=all_right.coords, dims=all_right.dims)

# 2. Create the visualization
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# TOP PANEL: Time-averaged Dissipation
# This collapses the time dimension to show the mean energy profile
env_left.mean(dim='time').plot(ax=axes[0], label='Left Side Mean Envelope', color='blue', lw=2)
env_right.mean(dim='time').plot(ax=axes[0], label='Right Side Mean Envelope', color='red', lw=2)

axes[0].set_title("Mean Energy Decay: Left vs Right")
axes[0].set_ylabel("Mean Amplitude")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# BOTTOM PANEL: The Difference Map
# This shows WHERE and WHEN the right side is losing more energy than the left
diff_env = env_left - env_right
diff_plot = diff_env.plot(ax=axes[1], x='x', y='time', cmap='Reds', add_colorbar=True)

axes[1].set_title("Energy Deficit (Left Envelope - Right Envelope)")
axes[1].set_ylabel("Time Step")
axes[1].set_xlabel("Spatial Coordinate (x)")

plt.tight_layout()
plt.show()

# 3. Simple Stat Check
total_dissipation = (1 - (env_right.sum() / env_left.sum())).values
print(f"Total energy loss on right side relative to left: {total_dissipation*100:.2f}%")

In [ ]:
import scipy.signal as signal

# Calculate PSD along the spatial dimension (x)
# This converts the 'x' axis into 'wavenumber'
freqs, psd_left = signal.welch(all_left, axis=1) # axis 1 is 'x'
_, psd_right = signal.welch(all_right, axis=1)

plt.figure(figsize=(10, 5))
plt.semilogy(freqs, psd_left.mean(axis=0), label='Left Side (Source)')
plt.semilogy(freqs, psd_right.mean(axis=0), label='Right Side (Dissipated)')
plt.xlabel('Wavenumber (1/spatial unit)')
plt.ylabel('Power Spectral Density')
plt.title('Energy Distribution by Scale')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 1. Calculate Total Energy (sum of envelope squared) for each time step
energy_left = (env_left**2).sum(dim='x')
energy_right = (env_right**2).sum(dim='x')

plt.figure(figsize=(10, 5))
energy_left.plot(label='Left Total Energy', color='blue')
energy_right.plot(label='Right Total Energy', color='red')
plt.title("Evolution of Total Energy over Time")
plt.xlabel("Time Step")
plt.ylabel("Integrated Energy (L2 Norm)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 2. Hovmöller comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), sharey=True)

env_left.plot(ax=ax1, x='x', y='time', cmap='viridis')
ax1.set_title("Left Side Spacetime (Envelope)")

env_right.plot(ax=ax2, x='x', y='time', cmap='viridis')
ax2.set_title("Right Side Spacetime (Envelope)")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch
from matplotlib.colors import LogNorm
import xarray as xr

# --- 1. Processing Function ---
def get_psd_over_time(da):
    """Computes PSD for each time step along the spatial dimension (x)."""
    # Calculate frequencies based on the 'x' dimension
    # Assumes 'x' is the second dimension (axis=1)
    freqs, _ = welch(da.isel(time=0).values)
    _, psd_values = welch(da.values, axis=1)
    
    return xr.DataArray(
        psd_values, 
        coords={'time': da.time, 'wavenumber': freqs}, 
        dims=['time', 'wavenumber']
    )

# --- 2. Compute the Spectral Data ---
psd_left = get_psd_over_time(all_left)
psd_right = get_psd_over_time(all_right)

# --- 3. Visualization Setup ---
# We define a 'floor' (vmin) to avoid log(0) errors
# and a shared 'vmax' so the colors are comparable.
v_min = 1e-10 
v_max = float(psd_left.max())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6), sharey=True)

# Plot Left Side
psd_left.where(psd_left > 0, v_min).plot(
    ax=ax1, 
    x='wavenumber', 
    y='time', 
    norm=LogNorm(vmin=v_min, vmax=v_max), 
    cmap='magma',
    cbar_kwargs={'label': 'Power Density'}
)
ax1.set_title("Left Side: Spectral Evolution", fontsize=14)
ax1.set_xlabel("Wavenumber (k)")
ax1.set_ylabel("Time Step")

# Plot Right Side
psd_right.where(psd_right > 0, v_min).plot(
    ax=ax2, 
    x='wavenumber', 
    y='time', 
    norm=LogNorm(vmin=v_min, vmax=v_max), 
    cmap='magma',
    cbar_kwargs={'label': 'Power Density'}
)
ax2.set_title("Right Side: Spectral Evolution", fontsize=14)
ax2.set_xlabel("Wavenumber (k)")
ax2.set_ylabel("") # Hide y-label since it's shared